# Week 4 — Member 1: Final Bengaluru Metro Network

**Purpose:** Build and validate the Metro network for the sustainable transport planner.

### Raw inputs used
1. `bengaluru_metro_network.csv` — prepared station/network data
2. `bengaluru_metro_stations.csv` — station reference data
3. `bmrcl.zip` — BMRCL GTFS data
4. `bmrcl-frequencies.zip` — BMRCL GTFS variant with frequency information
5. `data_dictionary.csv` — documentation for the prepared network dataset

### Main Week 4 outputs
- `metro_network.csv`
- `station_lookup.csv`
- `metro_routes.csv`
- `metro_transfers.csv`

The raw files are never modified.

In [1]:
import pandas as pd
from pathlib import Path
import zipfile
import io

print("Pandas version:", pd.__version__)

Pandas version: 3.0.6


## 1. Project paths

In [2]:
# Expected project structure
RAW_DIR = Path("data/raw")
OUTPUT_DIR = Path("data/processed/metro")

NETWORK_RAW = RAW_DIR / "bengaluru_metro_network.csv"
STATIONS_RAW = RAW_DIR / "bengaluru_metro_stations.csv"
BMRCL_ZIP = RAW_DIR / "bmrcl.zip"
BMRCL_FREQ_ZIP = RAW_DIR / "bmrcl-frequencies.zip"
DATA_DICTIONARY = RAW_DIR / "data_dictionary.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NETWORK_OUTPUT = OUTPUT_DIR / "metro_network.csv"
STATION_OUTPUT = OUTPUT_DIR / "station_lookup.csv"
ROUTES_OUTPUT = OUTPUT_DIR / "metro_routes.csv"
TRANSFERS_OUTPUT = OUTPUT_DIR / "metro_transfers.csv"

for path in [
    NETWORK_RAW, STATIONS_RAW, BMRCL_ZIP,
    BMRCL_FREQ_ZIP, DATA_DICTIONARY
]:
    print(f"{path}: {'FOUND' if path.exists() else 'MISSING'}")

data\raw\bengaluru_metro_network.csv: FOUND
data\raw\bengaluru_metro_stations.csv: FOUND
data\raw\bmrcl.zip: FOUND
data\raw\bmrcl-frequencies.zip: FOUND
data\raw\data_dictionary.csv: FOUND


## 2. Load the prepared Metro network

This file is the main source for station sequence, next station, line, interchange, coordinates and distance.

In [3]:
df = pd.read_csv(NETWORK_RAW)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("\nColumns:")
print(df.columns.tolist())

display(df.head())

Rows: 85
Columns: 10

Columns:
['station_code', 'station_name', 'line', 'sequence', 'is_interchange', 'next_station_code', 'latitude', 'longitude', 'distance_to_next_km', 'line_color']


,station_code,station_name,line,sequence,is_interchange,next_station_code,latitude,longitude,distance_to_next_km,line_color
0,WHTM,Whitefield (Kadugodi),Purple Line,1,0,UWVL,12.995699,77.757730,1.04,#7E22CE
1,UWVL,Hopefarm Channasandra,Purple Line,2,0,KDGD,12.987288,77.753629,0.79,#7E22CE
2,KDGD,Kadugodi Tree Park,Purple Line,3,0,ITPL,12.985649,77.746494,0.97,#7E22CE
3,ITPL,Pattandur Agrahara,Purple Line,4,0,SSHP,12.987593,77.737740,1.33,#7E22CE
4,SSHP,Sri Sathya Sai Hospital,Purple Line,5,0,VDHP,12.981164,77.727343,0.59,#7E22CE


## 3. Validate the prepared network

In [4]:
required_columns = [
    "station_code", "station_name", "line", "sequence",
    "is_interchange", "next_station_code", "latitude",
    "longitude", "distance_to_next_km", "line_color"
]

missing_columns = [c for c in required_columns if c not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("✓ Required columns present.")

print("\nDuplicate rows:", df.duplicated().sum())
print("\nDuplicate station codes:",
      df["station_code"].duplicated().sum())

print("\nMissing values:")
display(df[required_columns].isnull().sum())

invalid_lat = ((df["latitude"] < -90) | (df["latitude"] > 90)).sum()
invalid_lon = ((df["longitude"] < -180) | (df["longitude"] > 180)).sum()

print("\nInvalid latitude:", invalid_lat)
print("Invalid longitude:", invalid_lon)

if invalid_lat or invalid_lon:
    raise ValueError("Invalid coordinates detected.")

✓ Required columns present.

Duplicate rows: 0

Duplicate station codes: 2

Missing values:


station_code           0
station_name           0
line                   0
sequence               0
is_interchange         0
next_station_code      3
latitude               0
longitude              0
distance_to_next_km    0
line_color             0
dtype: int64


Invalid latitude: 0
Invalid longitude: 0


## 4. Load the station reference file

`bengaluru_metro_stations.csv` is used as a second station reference and validation source.

In [5]:
stations_ref = pd.read_csv(STATIONS_RAW)

print("Rows:", len(stations_ref))
print("Columns:", len(stations_ref.columns))
print(stations_ref.columns.tolist())

display(stations_ref.head())

Rows: 83
Columns: 3
['station_name', 'station_code', 'line']


,station_name,station_code,line
0,Attiguppe,AGPP,Purple Line
1,BTM Layout,BTML,Yellow Line
2,Baiyappanahalli,BYPL,Purple Line
3,Banashankari,BSNK,Green Line
4,Benniganahalli,BENN,Purple Line


In [6]:
# Find likely station-code and station-name columns automatically
def find_column(columns, candidates):
    lower_map = {str(c).lower(): c for c in columns}
    for candidate in candidates:
        if candidate in lower_map:
            return lower_map[candidate]
    return None

ref_code_col = find_column(
    stations_ref.columns,
    ["station_code", "stop_id", "station_id", "code"]
)

ref_name_col = find_column(
    stations_ref.columns,
    ["station_name", "stop_name", "name"]
)

print("Station code column:", ref_code_col)
print("Station name column:", ref_name_col)

if ref_code_col is None:
    raise ValueError("Could not identify a station-code column in bengaluru_metro_stations.csv")

# Compare codes where possible
network_codes = set(df["station_code"].astype(str))
reference_codes = set(stations_ref[ref_code_col].astype(str))

print("\nNetwork station codes:", len(network_codes))
print("Reference station codes:", len(reference_codes))
print("Codes in network but not reference:", len(network_codes - reference_codes))
print("Codes in reference but not network:", len(reference_codes - network_codes))

Station code column: station_code
Station name column: station_name

Network station codes: 83
Reference station codes: 83
Codes in network but not reference: 0
Codes in reference but not network: 0


## 5. Inspect the data dictionary

The dictionary is documentation only; it is not treated as a network table.

In [10]:
from pathlib import Path

dictionary_path = Path("data/raw/data_dictionary.csv")

print("Current folder:")
print(Path.cwd())

print("\nDoes the file exist?")
print(dictionary_path.exists())

print("\nFiles in data/raw:")
for file in Path("data/raw").glob("*"):
    print(file)

Current folder:
d:\Projects-2024-28\STA

Does the file exist?
True

Files in data/raw:
data\raw\bengaluru_metro_network.csv
data\raw\bengaluru_metro_stations.csv
data\raw\bmrcl-frequencies.zip
data\raw\bmrcl.zip
data\raw\BMRCL_COPY
data\raw\data_dictionary.csv


In [12]:
from pathlib import Path

dictionary_path = Path("data/raw/data_dictionary.csv")

with open(
    dictionary_path,
    "r",
    encoding="utf-8-sig",
    errors="replace"
) as f:
    lines = f.readlines()

print("Number of lines:", len(lines))
print("\nActual file contents:\n")

for i, line in enumerate(lines, start=1):
    print(f"{i}: {repr(line)}")

Number of lines: 11

Actual file contents:

1: 'column_name,description\n'
2: 'station_name,Name of the metro station\n'
3: 'station_code,Unique identifier code for each station\n'
4: 'line,Metro line name (Purple Line, Green Line, Yellow Line)\n'
5: 'sequence,Order of the station in a specific metro line\n'
6: 'is_interchange,Indicates whether the station is an interchange (1 = Yes, 0 = No)\n'
7: 'next_station_code,Code of the next station in the route (NULL if terminal station)\n'
8: 'latitude,Latitude coordinate of the station\n'
9: 'longitude,Longitude coordinate of the station\n'
10: 'distance_to_next_km,Distance between current station and next station in kilometers\n'
11: 'line_color,Hex color code representing the metro line'


In [13]:
from pathlib import Path
import csv

dictionary_path = Path("data/raw/data_dictionary.csv")

dictionary_rows = []

with open(
    dictionary_path,
    "r",
    encoding="utf-8-sig",
    errors="replace"
) as f:

    lines = f.readlines()

# First line is the header
for line in lines[1:]:
    line = line.rstrip("\n\r")

    # Split only at the FIRST comma
    column_name, description = line.split(",", 1)

    dictionary_rows.append({
        "column_name": column_name,
        "description": description
    })

dictionary = pd.DataFrame(dictionary_rows)

print("Rows:", len(dictionary))
print("Columns:", len(dictionary.columns))
print("Column names:", dictionary.columns.tolist())

display(dictionary)

Rows: 10
Columns: 2
Column names: ['column_name', 'description']


,column_name,description
0,station_name,Name of the metro station
1,station_code,Unique identifier code for each station
2,line,"Metro line name (Purple Line, Green Line, Yell..."
3,sequence,Order of the station in a specific metro line
4,is_interchange,Indicates whether the station is an interchang...
5,next_station_code,Code of the next station in the route (NULL if...
6,latitude,Latitude coordinate of the station
7,longitude,Longitude coordinate of the station
8,distance_to_next_km,Distance between current station and next stat...
9,line_color,Hex color code representing the metro line


In [ ]:
# with open(
#     "data/raw/data_dictionary.csv",
#     "r",
#     encoding="utf-8-sig",
#     errors="replace"
# ) as f:

#     lines = f.readlines()

# print("Number of lines:", len(lines))

# for i, line in enumerate(lines[:15], start=1):
#     print(i, repr(line))

Number of lines: 11
1 'column_name,description\n'
2 'station_name,Name of the metro station\n'
3 'station_code,Unique identifier code for each station\n'
4 'line,Metro line name (Purple Line, Green Line, Yellow Line)\n'
5 'sequence,Order of the station in a specific metro line\n'
6 'is_interchange,Indicates whether the station is an interchange (1 = Yes, 0 = No)\n'
7 'next_station_code,Code of the next station in the route (NULL if terminal station)\n'
8 'latitude,Latitude coordinate of the station\n'
9 'longitude,Longitude coordinate of the station\n'
10 'distance_to_next_km,Distance between current station and next station in kilometers\n'
11 'line_color,Hex color code representing the metro line'


## 6. Inspect both BMRCL GTFS ZIP files

The ZIP files are kept intact in `data/raw/`. We read their tables directly without extracting or modifying the originals.

In [14]:
def list_gtfs_files(zip_path):
    with zipfile.ZipFile(zip_path, "r") as z:
        return z.namelist()

print("bmrcl.zip:")
print(list_gtfs_files(BMRCL_ZIP))

print("\nbmrcl-frequencies.zip:")
print(list_gtfs_files(BMRCL_FREQ_ZIP))

bmrcl.zip:
['agency.txt', 'feed_info.txt', 'stops.txt', 'shapes.txt', 'routes.txt', 'trips.txt', 'stop_times.txt', 'fare_attributes.txt', 'fare_rules.txt', 'transfers.txt', 'levels.txt', 'pathways.txt', 'attributions.txt', 'translations.txt', 'calendar.txt', 'calendar_dates.txt', 'fare_media.txt', 'fare_products.txt', 'fare_leg_rules.txt', 'timeframes.txt', 'areas.txt', 'stop_areas.txt']

bmrcl-frequencies.zip:
['agency.txt', 'feed_info.txt', 'stops.txt', 'shapes.txt', 'routes.txt', 'trips.txt', 'stop_times.txt', 'fare_attributes.txt', 'fare_rules.txt', 'transfers.txt', 'levels.txt', 'pathways.txt', 'attributions.txt', 'translations.txt', 'calendar.txt', 'calendar_dates.txt', 'fare_media.txt', 'fare_products.txt', 'fare_leg_rules.txt', 'timeframes.txt', 'areas.txt', 'stop_areas.txt', 'frequencies.txt']


## 7. Read selected GTFS tables

For the Week 4 network, we use GTFS `routes.txt` and `transfers.txt` when available.

The frequency ZIP is inspected for `frequencies.txt`; its frequency records are kept separate because they are schedule/headway information rather than station-to-station topology.

In [15]:
def read_csv_from_zip(zip_path, filename):
    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()
        if filename not in names:
            return None
        with z.open(filename) as f:
            return pd.read_csv(f)

gtfs_routes = read_csv_from_zip(BMRCL_ZIP, "routes.txt")
gtfs_transfers = read_csv_from_zip(BMRCL_ZIP, "transfers.txt")
gtfs_stops = read_csv_from_zip(BMRCL_ZIP, "stops.txt")
gtfs_stop_times = read_csv_from_zip(BMRCL_ZIP, "stop_times.txt")

freq_routes = read_csv_from_zip(BMRCL_FREQ_ZIP, "routes.txt")
freq_transfers = read_csv_from_zip(BMRCL_FREQ_ZIP, "transfers.txt")
freq_stops = read_csv_from_zip(BMRCL_FREQ_ZIP, "stops.txt")
frequencies = read_csv_from_zip(BMRCL_FREQ_ZIP, "frequencies.txt")

for name, table in [
    ("GTFS routes", gtfs_routes),
    ("GTFS transfers", gtfs_transfers),
    ("GTFS stops", gtfs_stops),
    ("GTFS stop_times", gtfs_stop_times),
    ("Frequency GTFS routes", freq_routes),
    ("Frequency GTFS transfers", freq_transfers),
    ("Frequency GTFS stops", freq_stops),
    ("frequencies.txt", frequencies),
]:
    print(f"{name}: {None if table is None else table.shape}")

GTFS routes: (3, 8)
GTFS transfers: (16, 6)
GTFS stops: (488, 11)
GTFS stop_times: (88906, 8)
Frequency GTFS routes: (3, 8)
Frequency GTFS transfers: (16, 6)
Frequency GTFS stops: (488, 11)
frequencies.txt: (226, 5)


## 8. Create the station lookup

The station lookup combines the prepared network's station information with a GTFS stop reference where possible.

GTFS stop IDs are retained as a reference; the project station code remains the primary routing identifier.

In [16]:
station_lookup = df[
    [
        "station_code", "station_name", "line", "sequence",
        "is_interchange", "latitude", "longitude"
    ]
].copy()

station_lookup = station_lookup.drop_duplicates(
    subset=["station_code"]
).reset_index(drop=True)

# Add a GTFS stop_id column if a useful station-name match is possible.
station_lookup["gtfs_stop_id"] = pd.NA

if gtfs_stops is not None and "stop_id" in gtfs_stops.columns and "stop_name" in gtfs_stops.columns:
    # Exact normalized name matching is conservative.
    gtfs_name_map = (
        gtfs_stops.assign(
            _name_key=gtfs_stops["stop_name"].astype(str).str.strip().str.lower()
        )
        .drop_duplicates("_name_key")
        .set_index("_name_key")["stop_id"]
    )

    name_key = (
        station_lookup["station_name"]
        .astype(str).str.strip().str.lower()
    )

    station_lookup["gtfs_stop_id"] = name_key.map(gtfs_name_map)

station_lookup.to_csv(STATION_OUTPUT, index=False)

print("✓ station_lookup.csv created")
print("Rows:", len(station_lookup))
print("GTFS stop IDs matched:", station_lookup["gtfs_stop_id"].notna().sum())

display(station_lookup.head())

✓ station_lookup.csv created
Rows: 83
GTFS stop IDs matched: 72


,station_code,station_name,line,sequence,is_interchange,latitude,longitude,gtfs_stop_id
0,WHTM,Whitefield (Kadugodi),Purple Line,1,0,12.995699,77.757730,WHTM
1,UWVL,Hopefarm Channasandra,Purple Line,2,0,12.987288,77.753629,UWVL
2,KDGD,Kadugodi Tree Park,Purple Line,3,0,12.985649,77.746494,KDGD
3,ITPL,Pattandur Agrahara,Purple Line,4,0,12.987593,77.737740,ITPL
4,SSHP,Sri Sathya Sai Hospital,Purple Line,5,0,12.981164,77.727343,SSHP


## 9. Build the Metro station-to-station network

This is the core Week 4 output.

Each row represents one direct connection. Both directions are included so the future route engine can travel in either direction.

In [17]:
df = df.sort_values(["line", "sequence"]).reset_index(drop=True)

connections = []

for _, station in df.iterrows():

    next_code = station["next_station_code"]

    if pd.isna(next_code):
        continue

    destination = df[df["station_code"] == next_code]

    if destination.empty:
        print(
            f"WARNING: next station {next_code} "
            f"not found for {station['station_code']}"
        )
        continue

    destination = destination.iloc[0]

    forward = {
        "from_station_code": station["station_code"],
        "from_station_name": station["station_name"],
        "to_station_code": destination["station_code"],
        "to_station_name": destination["station_name"],
        "line": station["line"],
        "line_color": station["line_color"],
        "from_sequence": station["sequence"],
        "to_sequence": destination["sequence"],
        "from_latitude": station["latitude"],
        "from_longitude": station["longitude"],
        "to_latitude": destination["latitude"],
        "to_longitude": destination["longitude"],
        "distance_km": station["distance_to_next_km"],
        "from_is_interchange": station["is_interchange"],
        "to_is_interchange": destination["is_interchange"]
    }

    connections.append(forward)

    reverse = {
        "from_station_code": destination["station_code"],
        "from_station_name": destination["station_name"],
        "to_station_code": station["station_code"],
        "to_station_name": station["station_name"],
        "line": station["line"],
        "line_color": station["line_color"],
        "from_sequence": destination["sequence"],
        "to_sequence": station["sequence"],
        "from_latitude": destination["latitude"],
        "from_longitude": destination["longitude"],
        "to_latitude": station["latitude"],
        "to_longitude": station["longitude"],
        "distance_km": station["distance_to_next_km"],
        "from_is_interchange": destination["is_interchange"],
        "to_is_interchange": station["is_interchange"]
    }

    connections.append(reverse)

network_df = pd.DataFrame(connections)

network_df = network_df.drop_duplicates(
    subset=["from_station_code", "to_station_code", "line"]
).reset_index(drop=True)

network_df = network_df.sort_values(
    ["line", "from_sequence", "to_sequence"]
).reset_index(drop=True)

network_df.to_csv(NETWORK_OUTPUT, index=False)

print("✓ metro_network.csv created")
print("Connections:", len(network_df))

display(network_df.head(10))

✓ metro_network.csv created
Connections: 164


,from_station_code,from_station_name,to_station_code,to_station_name,line,line_color,from_sequence,to_sequence,from_latitude,from_longitude,to_latitude,to_longitude,distance_km,from_is_interchange,to_is_interchange
0,MDVA,Madavara,CKBL,Chikkabidarakallu,Green Line,#16A34A,1,2,13.057306,77.472845,13.052349,77.487821,1.71,0,0
1,CKBL,Chikkabidarakallu,MDVA,Madavara,Green Line,#16A34A,2,1,13.052349,77.487821,13.057306,77.472845,1.71,0,0
2,CKBL,Chikkabidarakallu,MJNN,Manjunath Nagar,Green Line,#16A34A,2,3,13.052349,77.487821,13.050310,77.494353,0.74,0,0
3,MJNN,Manjunath Nagar,CKBL,Chikkabidarakallu,Green Line,#16A34A,3,2,13.050310,77.494353,13.052349,77.487821,0.74,0,0
4,MJNN,Manjunath Nagar,NGSA,Nagasandra,Green Line,#16A34A,3,4,13.050310,77.494353,13.048236,77.500163,0.67,0,0
5,NGSA,Nagasandra,MJNN,Manjunath Nagar,Green Line,#16A34A,4,3,13.048236,77.500163,13.050310,77.494353,0.67,0,0
6,NGSA,Nagasandra,DSH,Dasarahalli,Green Line,#16A34A,4,5,13.048236,77.500163,13.043630,77.512324,1.41,0,0
7,DSH,Dasarahalli,NGSA,Nagasandra,Green Line,#16A34A,5,4,13.043630,77.512324,13.048236,77.500163,1.41,0,0
8,DSH,Dasarahalli,JLHL,Jalahalli,Green Line,#16A34A,5,6,13.043630,77.512324,13.039833,77.519939,0.93,0,0
9,JLHL,Jalahalli,DSH,Dasarahalli,Green Line,#16A34A,6,5,13.039833,77.519939,13.043630,77.512324,0.93,0,0


## 10. Create the Metro route reference from GTFS

`routes.txt` describes the transit services. We keep it as a separate processed reference instead of mixing schedule fields into the station topology.

In [18]:
if gtfs_routes is not None:
    metro_routes = gtfs_routes.copy()

    metro_routes.to_csv(
        ROUTES_OUTPUT,
        index=False
    )

    print("✓ metro_routes.csv created")
    print("Rows:", len(metro_routes))
    display(metro_routes.head())
else:
    metro_routes = pd.DataFrame()
    print("routes.txt not available in bmrcl.zip")

✓ metro_routes.csv created
Rows: 3


,route_long_name,route_short_name,agency_id,route_desc,route_type,route_id,route_color,route_text_color
0,Madavara - Silk Institute,Green,BMRCL,"Green Line passes through Madavara, Majestic a...",1,GREEN,009933,FFFFFF
1,Rashtreeya Vidyalaya Road - Delta Electronics ...,Yellow,BMRCL,Yellow Line passes through Rashtreeya Vidyalay...,1,YELLOW,FFDF00,NaN
2,Whitefield (Kadugodi) - Challaghatta,Purple,BMRCL,Purple Line passes through Whitefield (Kadugod...,1,PURPLE,8C2877,FFFFFF


## 11. Create the Metro transfer reference from GTFS

`transfers.txt` is kept separately because transfers are not ordinary adjacent station connections.

In [19]:
if gtfs_transfers is not None:
    metro_transfers = gtfs_transfers.copy()

    metro_transfers.to_csv(
        TRANSFERS_OUTPUT,
        index=False
    )

    print("✓ metro_transfers.csv created")
    print("Rows:", len(metro_transfers))
    display(metro_transfers.head())
else:
    metro_transfers = pd.DataFrame()
    print("transfers.txt not available in bmrcl.zip")

✓ metro_transfers.csv created
Rows: 16


,from_stop_id,to_stop_id,from_route_id,to_route_id,transfer_type,min_transfer_time
0,KGWA_PF1,KGWA_PF3,PURPLE,GREEN,2,60
1,KGWA_PF1,KGWA_PF4,PURPLE,GREEN,2,60
2,KGWA_PF2,KGWA_PF3,PURPLE,GREEN,2,60
3,KGWA_PF2,KGWA_PF4,PURPLE,GREEN,2,60
4,KGWA_PF3,KGWA_PF1,GREEN,PURPLE,2,60


## 12. Check frequency information

The frequency GTFS is used as a **reference for later travel-time/headway work**. It is not merged into the Week 4 station topology.

In [20]:
if frequencies is not None:
    print("Frequency records:", len(frequencies))
    print("Frequency columns:")
    print(frequencies.columns.tolist())
    display(frequencies.head())
else:
    print("frequencies.txt not available.")

Frequency records: 226
Frequency columns:
['trip_id', 'start_time', 'end_time', 'headway_secs', 'exact_times']


,trip_id,start_time,end_time,headway_secs,exact_times
0,t1002,10:28:00,10:47:00,660,1
1,t1002,10:47:00,16:01:00,480,1
2,t1002,16:01:00,16:28:30,330,1
3,t1076,19:30:00,20:00:00,660,1
4,t1076,20:00:00,20:32:00,480,1


## 13. Test: determine directly connected stations

In [21]:
def get_next_stations(station_code):
    result = network_df[
        network_df["from_station_code"] == station_code
    ]

    if result.empty:
        return pd.DataFrame(
            columns=[
                "to_station_code",
                "to_station_name",
                "line",
                "distance_km"
            ]
        )

    return result[
        [
            "to_station_code",
            "to_station_name",
            "line",
            "distance_km"
        ]
    ].reset_index(drop=True)

test_station = df.iloc[0]["station_code"]

print("Test station:", test_station)
print("Station name:", df.iloc[0]["station_name"])
print("\nDirectly connected stations:")

display(get_next_stations(test_station))

Test station: MDVA
Station name: Madavara

Directly connected stations:


,to_station_code,to_station_name,line,distance_km
0,CKBL,Chikkabidarakallu,Green Line,1.71


## 14. Network integrity check

In [22]:
valid_codes = set(df["station_code"].astype(str))

invalid_from = network_df[
    ~network_df["from_station_code"].astype(str).isin(valid_codes)
]

invalid_to = network_df[
    ~network_df["to_station_code"].astype(str).isin(valid_codes)
]

duplicate_edges = network_df.duplicated(
    subset=["from_station_code", "to_station_code", "line"]
).sum()

print("Invalid FROM stations:", len(invalid_from))
print("Invalid TO stations  :", len(invalid_to))
print("Duplicate edges      :", duplicate_edges)

if len(invalid_from) == 0 and len(invalid_to) == 0 and duplicate_edges == 0:
    print("\n✓ Network integrity check PASSED.")
else:
    print("\n⚠ Network integrity check found issues.")

Invalid FROM stations: 0
Invalid TO stations  : 0
Duplicate edges      : 0

✓ Network integrity check PASSED.


## 15. Final Week 4 summary

In [23]:
interchange_count = (
    df["is_interchange"]
    .astype(str)
    .str.lower()
    .isin(["true", "yes", "1"])
    .sum()
)

print("=" * 65)
print("MEMBER 1 — WEEK 4 METRO NETWORK COMPLETE")
print("=" * 65)

print("\nStations:", df["station_code"].nunique())
print("Metro lines:", df["line"].nunique())
print("Interchange stations:", interchange_count)
print("Network connections:", len(network_df))

print("\nGenerated processed files:")
for path in [
    NETWORK_OUTPUT,
    STATION_OUTPUT,
    ROUTES_OUTPUT,
    TRANSFERS_OUTPUT
]:
    print(" -", path)

print("\nRaw files remain unchanged.")
print("\nReady for Week 5 route-engine integration.")

MEMBER 1 — WEEK 4 METRO NETWORK COMPLETE

Stations: 83
Metro lines: 3
Interchange stations: 4
Network connections: 164

Generated processed files:
 - data\processed\metro\metro_network.csv
 - data\processed\metro\station_lookup.csv
 - data\processed\metro\metro_routes.csv
 - data\processed\metro\metro_transfers.csv

Raw files remain unchanged.

Ready for Week 5 route-engine integration.


## Final project structure

After running the notebook, your relevant Metro section should look like:

```text
data/
├── raw/
│   ├── bengaluru_metro_network.csv
│   ├── bengaluru_metro_stations.csv
│   ├── bmrcl.zip
│   ├── bmrcl-frequencies.zip
│   └── data_dictionary.csv
│
└── processed/
    └── metro/
        ├── metro_network.csv
        ├── station_lookup.csv
        ├── metro_routes.csv
        └── metro_transfers.csv

member1_metro_network_final.ipynb
```

`frequencies.txt` remains inside `bmrcl-frequencies.zip` and is inspected for later headway/travel-time work rather than duplicated into the Week 4 topology output.